# Task 2 - A Transformer that basecalls nanopore signals

A nanopore reads DNA by measuring an electric **current** as the strand passes through a
pore. The bases inside the pore set the current, so a read comes out as a noisy 1-D signal
(a *squiggle*). **Basecalling** inverts it: squiggle -> `ACGT`.

This notebook is built as an investigation, and the order matters. You will look at the
data, build the model you already know from Task 1, find out where it fails, work out why,
and only then reach for a Transformer. Work through it in order.

See the assignment PDF for the steps (A, B, ...) and the report questions.

## 0. Setup

In [ ]:
import time, numpy as np, torch
from torch import nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASES = "ACGT"
print("device:", DEVICE)

## 1. Look at the data (given)

The "pore model" is a frozen table: each **k-mer** (the `K` bases in the pore) maps to one
current level. To make a read we draw a random DNA template, look up each base's level to
get a staircase, hold each level for `DWELL` samples, and add Gaussian **noise**.

`NOISE` is your main data slider. The generator also has `hetero` (noise that varies along
the read) and `dwell_jitter` (a variable number of samples per base), which you will use in
Section 7.

**Read this code before moving on.** It is the ground truth for everything that follows,
and it is doing something to the reads that you will need to notice.

In [ ]:
K, DWELL, READ_LEN = 3, 8, 60
NOISE = 0.20                              # <-- DATA SLIDER (you will vary this in Sec. 6)
NUM_KMERS = 4 ** K
TOTAL = READ_LEN * DWELL                  # signal length (fixed, even with dwell jitter)
GRID = np.arange(1, READ_LEN) * DWELL     # the even base boundaries (8, 16, ...)
COMP = np.array([3, 2, 1, 0])             # complement: A<->T, C<->G

rng = np.random.default_rng(42)
KMER_LEVEL = rng.normal(0, 1, NUM_KMERS).astype(np.float32)   # the pore model

def kmer_ids(seqs):
    B, n = seqs.shape
    p = np.pad(seqs, ((0, 0), (K - 1, 0)), mode="wrap")
    ids = np.zeros((B, n), dtype=np.int64)
    for j in range(K):
        ids = ids * 4 + p[:, j:j + n]
    return ids

def make_batch(batch, noise=None, gen=None, return_marker=False,
               hetero=0.0, dwell_jitter=0.0):
    '''signal (B, TOTAL), label (B, READ_LEN), orientation (B,), [marker base idx].

    A read is one strand of DNA pulled through the pore. Half the time it is the
    reference strand, half the time it is that strand's reverse complement. The
    current always comes from whichever strand is actually in the pore. The answer
    we want is always the reference strand.

    Three data sliders:
      noise        - Gaussian noise strength on the current
      hetero       - if >0, the noise level VARIES per base (in [1-h, 1+h] * noise)
      dwell_jitter - if >0, each base emits a variable # of samples (boundaries wobble);
                     total length stays TOTAL, so the tokenizer is unchanged
    '''
    g = gen or rng; noise = NOISE if noise is None else noise
    templ = g.integers(0, 4, size=(batch, READ_LEN))           # the reference strand
    fwd = g.integers(0, 2, size=batch)                         # 1 = forward, 0 = reverse
    # whichever strand is physically in the pore is what sets the current
    in_pore = np.where(fwd[:, None] == 1, templ, COMP[templ][:, ::-1])
    lvl = KMER_LEVEL[kmer_ids(in_pore)]                        # (B, READ_LEN) level per base
    if dwell_jitter > 0:                                       # wobble the base boundaries
        bnd = GRID[None, :] + dwell_jitter * DWELL * g.normal(size=(batch, READ_LEN - 1))
        bnd = np.sort(np.clip(np.round(bnd), 1, TOTAL - 1), axis=1).astype(int)
        full = np.concatenate([np.zeros((batch, 1), int), bnd,
                               np.full((batch, 1), TOTAL, int)], axis=1)
    else:
        full = (np.arange(READ_LEN + 1) * DWELL)[None, :].repeat(batch, 0)
    t = np.arange(TOTAL)
    base_idx = np.clip((full[:, 1:, None] <= t[None, None, :]).sum(1), 0, READ_LEN - 1)
    sig = np.take_along_axis(lvl, base_idx, axis=1).astype(np.float32)     # (B, TOTAL)
    marker = np.where(fwd == 0, g.integers(0, READ_LEN, size=batch), -1)
    sig += 6.0 * ((base_idx == marker[:, None]) & (fwd[:, None] == 0))     # adapter spike
    if hetero > 0:
        bscale = 1 + hetero * (2 * g.random((batch, READ_LEN)) - 1)
        psc = np.take_along_axis(bscale, base_idx, axis=1)
        sig += (g.normal(0, 1, sig.shape) * noise * psc).astype(np.float32)
    else:
        sig += g.normal(0, noise, size=sig.shape).astype(np.float32)
    label = templ                       # always report the reference strand
    out = [torch.from_numpy(sig), torch.from_numpy(label.copy()), torch.from_numpy(fwd)]
    if return_marker: out.append(torch.from_numpy(marker))
    return tuple(out)

### Plot a forward read and a reverse read

Reads come off the machine in one of two orientations, and `make_batch` tells you which is
which. Run this a few times and compare the two panels carefully.

**Something in the signal distinguishes them.** Find it, and check whether it appears in
the same place on every read you draw. You will need this in Section 3.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
for ax, want in zip(axes, [1, 0]):
    while True:
        sig, lab, fwd = make_batch(1)
        if fwd[0] == want: break
    ax.plot(sig[0].numpy(), lw=.8)
    for i in range(READ_LEN): ax.axvline(i * DWELL, color="k", ls=":", alpha=.15)
    ax.set_title("FORWARD" if want else "REVERSE")
    ax.set_ylabel("current")
axes[-1].set_xlabel("signal sample"); plt.tight_layout(); plt.show()

### What the model has to output

The signal is built the same way in both cases: the bases in the pore set the current.
What changes is **which strand was in the pore**.

Half the reads are the reference strand itself. The other half are that strand's
**reverse complement**, which is what you get when the other side of the double helix
goes through: every base is complemented (A-T, C-G) *and* the order is flipped. Either
way, the answer we want is the **reference strand**.

A six-base example, to make the bookkeeping concrete:

```
reference strand (the answer we want)     A C G T A C

FORWARD read - the pore holds             A C G T A C
  answer position 0 comes from signal chunk 0, 1 from 1, ...     (straight across)

REVERSE read - the pore holds             G T A C G T      (reverse complement)
  answer position 0 comes from signal chunk 5, 1 from 4, ...     (mirrored)
```

So on a reverse read, the base you have to name at the **start** of the read is
determined by current at the **end** of it.

Which leaves one problem: the two strands are statistically identical, so the *pore
levels alone* cannot tell you which case you are looking at. Something else in the signal
has to, and that is what you were hunting for in the plot above.

## 2. First attempt: the CNN you already know (TODO-1)

Start with the architecture you built in Task 1. The task is different - one label per
*base* rather than one per sequence - but the ingredients are the same.

The structure below mirrors the Task 1 CNN one piece at a time:

| what it does | layer |
|---|---|
| turn each base's `DWELL` signal samples into one token of width `C` | a strided `Conv1d` (given) |
| mix each token with its **neighbours** | `Conv1d` + `ReLU`, repeated `layers` times |
| turn each token into its 4 base scores | `Conv1d(C, 4, 1)` |

Two things worth knowing before you write it:

- `nn.Conv1d(C, C, kernel, padding=kernel//2)` keeps the length unchanged, so the shape
  stays `(batch, C, READ_LEN)` all the way through the body.
- A `Conv1d` with `kernel_size=1` is a `Linear` applied to every position separately. It is
  how you get 4 scores per base while keeping the `(batch, channels, position)` layout that
  `Conv1d` uses.

`forward` is written for you, including the transposes, so you only fill in the two layer
groups.

In [ ]:
class CNNBasecaller(nn.Module):
    """Same task as Basecaller, but local convolutions instead of global attention."""
    def __init__(self, C=112, layers=3, kernel=5):
        super().__init__()
        # given: the same tokenizer trick - one token per base
        self.stem = nn.Conv1d(1, C, kernel_size=DWELL, stride=DWELL)

        # TODO-1a: the body. `layers` repetitions of Conv1d(C, C, kernel, padding=kernel//2)
        #          followed by nn.ReLU(). Wrap them in nn.Sequential(*...) so they run in
        #          order. Shape stays (batch, C, READ_LEN) throughout.
        self.body = ...

        # TODO-1b: the head. One Conv1d turning C channels into 4, with kernel_size=1.
        self.head = ...

    def forward(self, sig):                    # sig (batch, READ_LEN*DWELL)
        x = self.stem(sig[:, None, :])         # -> (batch, C, READ_LEN)
        x = self.body(x)                       # -> (batch, C, READ_LEN)
        x = self.head(x)                       # -> (batch, 4, READ_LEN)
        return x.transpose(1, 2)               # -> (batch, READ_LEN, 4)



### A note on the loss (given - no new math)

In Task 1 the model produced **one** score per sequence, a `sigmoid` turned it into "probability of binding", and `BCEWithLogitsLoss` measured how wrong it was.

Here the question is not yes/no but **which of four** - so the model produces **four** scores per base. The four-way version of that same pair is `softmax` (turns the four scores into four probabilities that sum to 1) and `nn.CrossEntropyLoss` (measures how wrong they are). Same idea, four choices instead of two.

You do not have to derive or implement it - `train` below already uses `nn.CrossEntropyLoss`, and, exactly like `BCEWithLogitsLoss`, it applies the softmax internally, so your model's head stays a plain `nn.Linear` with **no softmax inside**. To read off a called base we just take the largest of the four scores (`.argmax(-1)`).


### Train it (given)

`evaluate` reports accuracy **split by orientation**. Watch the two columns, not just the
overall number.

In [ ]:
def evaluate(model, n=4000, seed=1234, **data):
    # fixed generator: the same model always scores the same, on the same reads
    model.eval()
    with torch.no_grad():
        sig, lab, fwd = make_batch(n, gen=np.random.default_rng(seed), **data)
        pred = model(sig.to(DEVICE)).argmax(-1).cpu()
    ok = (pred == lab).float().mean(1)
    return (pred == lab).float().mean().item(), ok[fwd == 1].mean().item(), ok[fwd == 0].mean().item()

def train(model, name="model", steps=4000, batch=128, lr=1.5e-3, seed=0, **data):
    # NOTE: seed your weights BEFORE building the model, e.g.
    #   torch.manual_seed(0); model = Basecaller(...)
    # calling it here only affects what happens after construction.
    # **data forwards the sliders (noise=, hetero=, dwell_jitter=) to make_batch
    torch.manual_seed(seed); model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.CrossEntropyLoss()
    gen = np.random.default_rng(7); t0 = time.time()
    for _ in range(steps):
        sig, lab, _ = make_batch(batch, gen=gen, **data)
        loss = lossf(model(sig.to(DEVICE)).reshape(-1, 4), lab.reshape(-1).to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
    acc, af, ar = evaluate(model, **data)          # evaluate in the SAME regime it trained on
    print(f"{name:16s} params={sum(p.numel() for p in model.parameters()):>8,}  "
          f"acc={acc:.3f}  (forward {af:.3f} | reverse {ar:.3f})  {time.time()-t0:.0f}s")
    return acc

In [ ]:
cnn = CNNBasecaller()
train(cnn, "CNN baseline")

## 3. What went wrong? (given)

One of those two columns is much worse than the other. Before you try to fix it, find out
*where* the model is failing.

The plot below takes reverse reads only and asks: how does the CNN's accuracy depend on how
far a base sits from the thing you spotted in Section 1?

In [ ]:
# uses the `cnn` you trained in the cell above
g = np.random.default_rng(0); n = 4000
templ = g.integers(0, 4, size=(n, READ_LEN))                 # the reference strand
in_pore = COMP[templ][:, ::-1]                               # reverse reads only
sig = np.repeat(KMER_LEVEL[kmer_ids(in_pore)], DWELL, axis=1).astype(np.float32)
qpos = g.integers(0, READ_LEN, size=n)
for b in range(n): sig[b, qpos[b]*DWELL:(qpos[b]+1)*DWELL] += 6.0
sig += g.normal(0, NOISE, size=sig.shape).astype(np.float32)
label = templ
cnn.eval()
with torch.no_grad():
    pred = cnn(torch.from_numpy(sig).to(DEVICE)).argmax(-1).cpu().numpy()
correct = (pred == label); dist = np.abs(np.arange(READ_LEN)[None, :] - qpos[:, None])
D = 40; acc = [correct[dist == d].mean() for d in range(D)]
plt.figure(figsize=(9, 2.8)); plt.plot(range(D), acc, marker=".")
plt.axhline(0.25, color="grey", ls="--", label="chance")
plt.title("CNN accuracy on reverse reads vs. distance from marker")
plt.xlabel("bases from marker"); plt.ylabel("accuracy"); plt.legend(); plt.tight_layout(); plt.show()

Think about what sets how far a convolutional filter can "see". Stacking
`layers` convolutions of width `kernel` gives each output a **receptive field** of roughly
`layers * (kernel - 1) + 1` bases, and nothing outside that window can influence it.

Compare that number with the shape of the curve you just plotted. Then convince yourself
that making the CNN wider (more channels) would not change the picture.

## 4. The fix: attention (TODO-2)

The problem is not capacity, it is **reach**. We need an architecture where any position can
draw on any other position, no matter how far apart they are. That is exactly what
**self-attention** does: every token compares itself with every other token and takes a
weighted sum of them all.

Below are five working blocks. `EncoderBlock` already combines attention and a per-token
feed-forward network, along with a few other things that keep a deep stack trainable. You do
not need to change anything inside them or understand their internals - you create them with
a size and use them.

> `MultiHeadSelfAttention` is written from scratch so you can see the mechanism
> `softmax(QK^T / sqrt(dk)) V` from the tutorial.

In [ ]:
class Tokenizer(nn.Module):
    "Collapse each base's DWELL signal samples into one token vector (a strided conv)."
    def __init__(self, d_model):
        super().__init__()
        self.conv = nn.Conv1d(1, d_model, kernel_size=DWELL, stride=DWELL)
    def forward(self, sig):                 # sig (B, READ_LEN*DWELL)
        return self.conv(sig[:, None, :]).transpose(1, 2)     # (B, READ_LEN, d_model)

class PositionalEncoding(nn.Module):
    "Add a fixed sinusoidal position signature so attention can tell positions apart."
    def __init__(self, d_model, max_len=READ_LEN):
        super().__init__()
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[None, :x.size(1)]

class MultiHeadSelfAttention(nn.Module):
    "Every token looks at every token: softmax(Q Kᵀ / sqrt(dk)) V, from scratch."
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x, return_attn=False):
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.h, self.dk).transpose(1, 2) for t in (q, k, v)]
        att = (q @ k.transpose(-2, -1) / self.dk ** 0.5).softmax(dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, T, d)
        return (self.out(y), att) if return_attn else self.out(y)

class FeedForward(nn.Module):
    "Per-token 2-layer MLP that mixes features (not positions)."
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    "One encoder: attention, then feed-forward - each wrapped in a residual + LayerNorm."
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # attention + residual connection
        x = x + self.ff(self.ln2(x))       # feed-forward + residual connection
        return x

**`TODO-2`.** Wire the blocks into the full model: a `Tokenizer`, a
`PositionalEncoding`, `n_layers` `EncoderBlock`s, and a `Linear` head. The constructor
arguments are your architecture decisions: `d_model` (token width), `n_heads`, `n_layers`,
`d_ff` (feed-forward width), and `use_pe` (positional encoding on/off, for Section 6).

In [ ]:
class Basecaller(nn.Module):
    def __init__(self, d_model=96, n_heads=4, n_layers=3, d_ff=128, use_pe=True):
        super().__init__()
        # TODO-2: build the stack from the bricks (these args are your design choices)
        #   self.tokenizer : Tokenizer(...)
        #   self.pos       : PositionalEncoding(...)  (only if use_pe)
        #   self.blocks    : nn.ModuleList of n_layers EncoderBlock(...)
        #   self.head      : nn.Linear(...) mapping each token to 4 base logits
        self.tokenizer = ...
        self.pos = ...
        self.blocks = ...
        self.head = ...
    def forward(self, sig):
        x = self.tokenizer(sig)
        if self.pos is not None: x = self.pos(x)
        for block in self.blocks: x = block(x)
        return self.head(x)                       # (B, READ_LEN, 4)

**`TODO-3`.** Instantiate your `Basecaller` with sizes you choose and
train it on the *same* data as the CNN. Compare the two accuracy columns with what the CNN
managed.

In [ ]:
# TODO-3: instantiate YOUR Basecaller (pick d_model / n_heads / n_layers / d_ff) and train it
model = Basecaller(...)
train(model, "Transformer")

## 5. Why did that work? (given)

Attention weights are **directly readable**, which is unusual for a neural network: we can
see which positions the model actually used. We take a **reverse** read and ask, when the
model computes base 30, where does it look?

**What is plotted.** Base 30's query is compared against every key, then softmaxed over the
keys, exactly as in the tutorial. These are the same weights that build base 30's output.

**How to read it**

- **x-axis** - the position being looked *at*.
- **y-axis** - the share of base 30's attention spent there. Each head's curve sums to 1
  across the read, so an even split would be a flat line at $1/60 \approx 0.017$. A point at
  0.25 means that head spends a quarter of its attention on that single base.
- **One line per head** - heads specialise, and averaging them mixes different jobs together.
- **Red dashed line** - where the thing you found in Section 1 actually is.

In [ ]:
# uses the `model` you trained in Section 4
while True:
    sig, lab, fwd, marker = make_batch(1, return_marker=True)
    if fwd[0] == 0: break
model.eval()
with torch.no_grad():
    x = model.tokenizer(sig.to(DEVICE)); x = model.pos(x) if model.pos else x
    _, att = model.blocks[0].attn(model.blocks[0].ln1(x), return_attn=True)

A = att[0, :, 30, :].cpu()                      # (n_heads, READ_LEN): base 30 -> every base
plt.figure(figsize=(9, 3))
for h in range(A.shape[0]):
    plt.plot(A[h], marker=".", lw=1.2, label=f"head {h}")
plt.axvline(int(marker[0]), color="r", ls="--", lw=2, label=f"marker @ {int(marker[0])}")
plt.title("Attention from base 30 across a reverse read (block 0, per head)")
plt.xlabel("base position"); plt.ylabel("attention weight")
plt.legend(fontsize=8, ncol=3); plt.tight_layout(); plt.show()

Compare this with the diagnosis in Section 3. Try `model.blocks[1]` and
`model.blocks[2]` as well: in our runs the middle block is mostly **local**, concentrating on
neighbouring bases, which is what you need to resolve the k-mer. So the network splits the
work across layers, fetching global information in one layer and reading bases locally in
another.

A spike away from both base 30 and the red line is not a bug, it is a different head doing a
different job.

**One honest caveat.** This orientation rule is a teaching device, not nanopore biology. A real basecaller outputs whatever strand passed through the pore and never complements anything: matching a read to the correct strand of a reference happens later, during alignment. We folded that step into the model, and marked it with a spike, because it creates one clean long-range dependency to study. In real reads the long-range structure comes from variable translocation speed and from homopolymer runs instead.


## 6. Ablation: turn positional encoding off (TODO-4)

Rebuild your model with `use_pe=False`, retrain, and compare. (Hint: what does
self-attention *not* know about the tokens without it?)

In [ ]:
# TODO-4: train a Basecaller with use_pe=False and compare
m = Basecaller(use_pe=False)
train(m, "no-PE")

## 7. Harder data, bigger models (TODO-5)

The generator has three ways to make the reads harder:

- **`noise`** - the strength of the Gaussian noise on the current
- **`hetero`** - the noise level *varies* along the read, so some stretches are clean and
  others are bad
- **`dwell_jitter`** - each base emits a *variable* number of samples, so the boundaries
  between bases wobble

and your model has capacity settings (`d_model`, `n_layers`, `n_heads`, `d_ff`). As the data
gets harder, bigger models pull ahead, and that gap is what the competitive part of the grade
rewards.

Sweep `noise` for a couple of model sizes and plot **accuracy vs noise**. Then train in the
harder regime you are ranked on: `noise=0.4, hetero=0.6, dwell_jitter=0.15`.

In [ ]:
# TODO-5: sweep noise (try a second model size too); collect accuracy for a plot
for noise in [0.3, 0.5, 0.7]:
    train(Basecaller(...), f"noise={noise}", noise=noise)

# then the harder regime you are ranked on:
train(Basecaller(...), "graded regime", noise=0.4, hetero=0.6, dwell_jitter=0.15)